In [1]:
from mlx_lm import load

# same checkpoint the pipeline actually uses, so this tests the real chat template
model, tokenizer = load("mlx-community/Qwen3-8B-4bit")

THINK_MARKER = "REASONING_MARKER_ONLY_IN_THINK_BLOCK"
ANSWER_MARKER = "FINAL_ANSWER_MARKER_ONLY_IN_PROOF"

# shaped like a real attempt() output: a <think> block followed by a ```lean4 block
fake_assistant_turn = (
    f"<think>\n{THINK_MARKER} — pretend reasoning about the proof strategy here.\n</think>\n\n"
    f"```lean4\n{ANSWER_MARKER}\ntheorem fake : True := by trivial\n```\n"
)

messages = [
    {"role": "user", "content": "Complete the following Lean 4 theorem using Mathlib.\n\n```lean4\ntheorem fake : True := by sorry```"},
    {"role": "assistant", "content": fake_assistant_turn},
    {"role": "user", "content": "Lean rejected this proof. Fix it and try again.\n"},
]

for thinking_flag in (True, False):
    # tokenize=False so we get the actual string back to read — production code
    # leaves tokenize at its default because it wants token ids straight into generate()
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=thinking_flag
    )
    print("=" * 25, f"enable_thinking={thinking_flag}", "=" * 25)
    print(rendered)
    print()
    print("think block survived in history:  ", THINK_MARKER in rendered)
    print("proof block survived in history:  ", ANSWER_MARKER in rendered)
    print()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

========================= enable_thinking=True =========================
<|im_start|>user
Complete the following Lean 4 theorem using Mathlib.

```lean4
theorem fake : True := by sorry```<|im_end|>
<|im_start|>assistant
```lean4
FINAL_ANSWER_MARKER_ONLY_IN_PROOF
theorem fake : True := by trivial
```
<|im_end|>
<|im_start|>user
Lean rejected this proof. Fix it and try again.
<|im_end|>
<|im_start|>assistant


think block survived in history:   False
proof block survived in history:   True

========================= enable_thinking=False =========================
<|im_start|>user
Complete the following Lean 4 theorem using Mathlib.

```lean4
theorem fake : True := by sorry```<|im_end|>
<|im_start|>assistant
```lean4
FINAL_ANSWER_MARKER_ONLY_IN_PROOF
theorem fake : True := by trivial
```
<|im_end|>
<|im_start|>user
Lean rejected this proof. Fix it and try again.
<|im_end|>
<|im_start|>assistant
<think>

</think>



think block survived in history:   False
proof block survived in history: 

In [1]:
import json
import wrapper

# --- minimal setup: no Mathlib needed, just Lean's core library ---
Repl = wrapper.LEANRepl()
Repl.import_project([], {})   # bare environment, no imports/opens

# the exact same construction attempt() does internally, pulled out so we can
# drive it by hand without needing the model
def identity_check_cmd(template, thm_name):
    return (
        template
        .replace(f"theorem {thm_name}", "example", 1)
        .replace("sorry", f"exact {thm_name}", 1)
    )

template = "theorem fake_thm (n : Nat) : n + 0 = n := by\n  sorry\n"
thm_name = template.split()[1]   # "fake_thm"

print("="*20, "case 1: a real, correct proof of the real statement", "="*20)
proof_1 = "theorem fake_thm (n : Nat) : n + 0 = n := by\n  simp\n"
answer_1 = Repl.write_proof(proof_1 + "\n\n" + identity_check_cmd(template, thm_name))
check_1 = Repl.check(answer_1)
print("check:", check_1, "  (expected: 1)")
print(answer_1)

print()
print("="*20, "case 2: a trivial restatement under the real name — the A1 hole", "="*20)
proof_2 = "theorem fake_thm : True := by trivial\n"
answer_2 = Repl.write_proof(proof_2 + "\n\n" + identity_check_cmd(template, thm_name))
check_2 = Repl.check(answer_2)
print("check:", check_2, "  (expected: -1, NOT 1 — this is the exact case A1 exists to catch)")
print(answer_2)

print()
print("="*20, "case 3: a REPL-level harness error — bad env id, not a proof error", "="*20)
bad_answer = Repl.send(json.dumps({"cmd": "example : True := trivial", "env": 999999}) + "\n\n")
check_3 = Repl.check(bad_answer)
print("check:", check_3, '  (expected: "harness_error")')
print(bad_answer)

Repl.close()

==================== case 1: a real, correct proof of the real statement ====================
check: -1   (expected: 1)
{'messages': [{'severity': 'error', 'pos': {'line': 6, 'column': 2}, 'endPos': {'line': 6, 'column': 16}, 'data': 'Type mismatch\n  fake_thm\nhas type\n  ∀ (n : Nat), n + 0 = n\nbut is expected to have type\n  n + 0 = n'}], 'env': 1}

==================== case 2: a trivial restatement under the real name — the A1 hole ====================
check: -1   (expected: -1, NOT 1 — this is the exact case A1 exists to catch)
{'messages': [{'severity': 'error', 'pos': {'line': 5, 'column': 2}, 'endPos': {'line': 5, 'column': 16}, 'data': 'Type mismatch\n  fake_thm\nhas type\n  True\nbut is expected to have type\n  n + 0 = n'}], 'env': 2}

==================== case 3: a REPL-level harness error — bad env id, not a proof error ====================
check: harness_error   (expected: "harness_error")
{'message': 'Unknown environment.'}


In [1]:
import retry_loop
import wrapper

class FakeTokenizer:
    def apply_chat_template(self, messages, add_generation_prompt=True, enable_thinking=True):
        return "unused — fake_generate ignores this"

def make_fake_generate(fixed_response):
    def fake_generate(model, tokenizer, prompt, max_tokens, verbose, sampler, logits_processors):
        return fixed_response
    return fake_generate

Repl = wrapper.LEANRepl()
Repl.import_project([], {})   # bare environment, no Mathlib needed

template = "theorem fake_thm (n : Nat) : n + 0 = n := by\n  sorry\n"
messages = [{"role": "user", "content": "doesn't matter for this test"}]
fake_tokenizer = FakeTokenizer()

cases = {
    "case 1: real, correct proof": "```lean4\ntheorem fake_thm (n : Nat) : n + 0 = n := by\n  simp\n```",
    "case 2: trivial restatement under the real name": "```lean4\ntheorem fake_thm : True := by trivial\n```",
}

for label, fake_raw_text in cases.items():
    retry_loop.generate = make_fake_generate(fake_raw_text)
    check, proof, raw_text, answer = retry_loop.attempt(
        messages, template, Repl, None, fake_tokenizer,
        enable_thinking=True, max_tokens=12000, verbose=True
    )
    print("="*20, label, "="*20)
    print("check:", check)
    print(answer)
    print()

# case 3: force a real harness failure by corrupting the env id, then go through attempt() normally
print("="*20, "case 3: REPL-level harness error", "="*20)
retry_loop.generate = make_fake_generate(cases["case 1: real, correct proof"])
Repl.base_env = 999999
check, proof, raw_text, answer = retry_loop.attempt(
    messages, template, Repl, None, fake_tokenizer,
    enable_thinking=True, max_tokens=12000, verbose=True
)
print("check:", check)
print(answer)

Repl.close()

{'env': 1}
==================== case 1: real, correct proof ====================
check: 1
{'env': 1}

{'messages': [{'severity': 'error', 'pos': {'line': 6, 'column': 2}, 'endPos': {'line': 6, 'column': 16}, 'data': 'Tactic `apply` failed: could not unify the type of `fake_thm`\n  True\nwith the goal\n  n + 0 = n\n\nn : Nat\n⊢ n + 0 = n'}], 'env': 2}
==================== case 2: trivial restatement under the real name ====================
check: -1
{'messages': [{'severity': 'error', 'pos': {'line': 6, 'column': 2}, 'endPos': {'line': 6, 'column': 16}, 'data': 'Tactic `apply` failed: could not unify the type of `fake_thm`\n  True\nwith the goal\n  n + 0 = n\n\nn : Nat\n⊢ n + 0 = n'}], 'env': 2}

==================== case 3: REPL-level harness error ====================
{'message': 'Unknown environment.'}
check: harness_error
{'message': 'Unknown environment.'}
